In [17]:
import pandas as pd
import itertools
from collections import Counter, defaultdict


In [18]:
df = pd.read_csv(r"C:\Users\lilit\PycharmProjects\nlp-course-2025.1\Lilit Mnatsakanyan\data\student_data.csv")
df = df.sort_values(["StudentID", "Day"]).reset_index(drop=True)


In [19]:
df

,StudentID,Day,Mood,ShirtColor
0,1,1,H,R
1,1,2,H,R
2,1,3,S,B
3,1,4,S,B
4,1,5,H,R
...,...,...,...,...
95,5,16,S,G
96,5,17,H,R
97,5,18,H,R
98,5,19,H,G


In [20]:
states = ["H", "S"]
obs_vals = ["R", "G", "B"]
test_obs = ["R", "B", "G"]
alpha = 1

In [21]:
first_days = df[df["Day"] == 1]
count_first = first_days["Mood"].value_counts().to_dict()
n_students = first_days["StudentID"].nunique()

In [22]:
pi = {}
for s in states:
    pi[s] = (count_first.get(s, 0) + alpha) / (n_students + alpha * len(states))


trans_counts = defaultdict(Counter)
source_counts = Counter()

In [23]:
for sid, g in df.groupby("StudentID"):
    g = g.sort_values("Day")
    moods = g["Mood"].tolist()
    for m1, m2 in zip(moods[:-1], moods[1:]):
        trans_counts[m1][m2] += 1
        source_counts[m1] += 1

A = {s: {} for s in states}
for s in states:
    denom = source_counts.get(s, 0) + alpha * len(states)
    for t in states:
        A[s][t] = (trans_counts[s][t] + alpha) / denom if denom > 0 else 0.0


In [24]:
emit_counts = defaultdict(Counter)
state_counts = Counter(df["Mood"])

In [25]:
for _, row in df.iterrows():
    emit_counts[row["Mood"]][row["ShirtColor"]] += 1

B = {s: {} for s in states}
for s in states:
    denom = state_counts.get(s, 0) + alpha * len(obs_vals)
    for o in obs_vals:
        B[s][o] = (emit_counts[s][o] + alpha) / denom if denom > 0 else 0.0


In [26]:
def joint_prob(seq, observations):
    m1, m2, m3 = seq
    r1, r2, r3 = observations
    return (
            pi[m1] *
            B[m1][r1] *
            A[m1][m2] *
            B[m2][r2] *
            A[m2][m3] *
            B[m3][r3]
    )


In [27]:
seqs = list(itertools.product(states, repeat=3))
rows = []
for seq in seqs:
    p = joint_prob(seq, test_obs)
    rows.append({"Sequence": "".join(seq), "Probability": p})

res = pd.DataFrame(rows).sort_values("Probability", ascending=False).reset_index(drop=True)


In [28]:
print("Initial distribution (pi):")
pd.Series(pi)

Initial distribution (pi):


H    0.571429
S    0.428571
dtype: float64

In [29]:
print("\nTransition matrix (A):")
pd.DataFrame(A).T


Transition matrix (A):


,H,S
H,0.649123,0.350877
S,0.452381,0.547619


In [30]:
print("\nEmission matrix (B):")
pd.DataFrame(B).T


Emission matrix (B):


,R,G,B
H,0.700000,0.283333,0.016667
S,0.021739,0.152174,0.826087


In [31]:
print("\nAll sequences with P(M1,M2,M3,R,B,G):")
print(res.to_string(index=False))


All sequences with P(M1,M2,M3,R,B,G):
Sequence  Probability
     HSH     0.014861
     HSS     0.009662
     HHH     0.000796
     SSH     0.000540
     SSS     0.000351
     HHS     0.000231
     SHH     0.000013
     SHS     0.000004


In [32]:
best = res.iloc[0]
print(f"\nMost likely sequence (argmax): {best['Sequence']} with probability {best['Probability']:.6g}")



Most likely sequence (argmax): HSH with probability 0.0148608
